<a href="https://colab.research.google.com/github/irissouza7/Mestrado-IDP_Publico/blob/main/Consolidacao_Arquivos_SINAPI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importar dados dos arquivos

In [ ]:
import pandas as pd
import os
from collections import Counter

def clean_column_names(df):
    # Get initial cleaned names, allowing temporary duplicates
    cleaned_names_temp = []
    for col in df.columns:
        cleaned_col = str(col).replace('\n', '').strip().lower().replace(' ', '_')
        cleaned_names_temp.append(cleaned_col)

    # Resolve duplicates by appending suffixes
    final_unique_cols = []
    counts = Counter(cleaned_names_temp)
    seen_counts = Counter() # To keep track of how many times a cleaned name has been added
    for col_name in cleaned_names_temp:
        seen_counts[col_name] += 1
        if counts[col_name] > 1:
            # If it's a duplicate, append a unique suffix based on its occurrence.
            # Example: 'col' -> 'col_1', 'col' -> 'col_2'
            final_unique_cols.append(f"{col_name}_{seen_counts[col_name]}")
        else:
            final_unique_cols.append(col_name)

    df.columns = final_unique_cols
    return df

In [ ]:

# Definição dos arquivos
#ARQUIVO_A = r"SINAPI_Referência_2025_01.xlsx"
ARQUIVO_B = r"SINAPI_referencia_consolidada_jan25_jul26.xlsx"


In [ ]:
from google.colab import files

# Clique no botão 'Choose Files' e selecione todos os arquivos Excel da lista.
print("Por favor, carregue os arquivos SINAPI_Referência_YYYY_MM.xlsx:")
files.upload()

Por favor, carregue os arquivos SINAPI_Referência_YYYY_MM.xlsx:


In [ ]:
# Lista de arquivos SINAPI a serem processados
LISTA_ARQUIVOS_SINAPI = [
    "SINAPI_Referência_2025_01.xlsx",
    "SINAPI_Referência_2025_02.xlsx",
    "SINAPI_Referência_2025_03.xlsx",
    "SINAPI_Referência_2025_04.xlsx",
    "SINAPI_Referência_2025_05.xlsx",
    "SINAPI_Referência_2025_06.xlsx",
    "SINAPI_Referência_2025_07.xlsx",
    "SINAPI_Referência_2025_08.xlsx",
    "SINAPI_Referência_2025_09.xlsx",
    "SINAPI_Referência_2025_10.xlsx",
    "SINAPI_Referência_2025_11.xlsx",
    "SINAPI_Referência_2025_12.xlsx",
    "SINAPI_Referência_2026_01.xlsx",
    "SINAPI_Referência_2026_02.xlsx",
    "SINAPI_Referência_2026_03.xlsx",
    "SINAPI_Referência_2026_04.xlsx",
    "SINAPI_Referência_2026_05.xlsx",
    "SINAPI_Referência_2026_06.xlsx",
    "SINAPI_Referência_2026_07.xlsx"
]

print(f"Lista de arquivos SINAPI definida: {len(LISTA_ARQUIVOS_SINAPI)} arquivos.")

Lista de arquivos SINAPI definida: 19 arquivos.


In [ ]:
all_consolidated_dfs = []

sheet_names_to_process = ["ISD", "ICD", "ISE"]

for file_path in LISTA_ARQUIVOS_SINAPI:
    print(f"Processando arquivo: {file_path}")

    # 1. Extrair ano_mes do nome do arquivo atual
    filename = os.path.basename(file_path)
    filename_without_ext = filename.replace(".xlsx", "")
    parts = filename_without_ext.split("_")

    current_ano_mes = None
    if len(parts) >= 4:
        year = parts[-2]
        month_part = parts[-1]
        month = month_part.split(' ')[0]
        current_ano_mes = f"{year}_{month}"
    else:
        print(f"Warning: Formato de nome de arquivo inesperado para {file_path}. Não foi possível extrair YYYY_MM.")
        continue # Pular para o próximo arquivo se o formato for inválido

    if not current_ano_mes:
        print(f"Erro: Não foi possível determinar ano_mes para {file_path}. Pulando este arquivo.")
        continue

    # 2. Ler todas as planilhas do arquivo atual
    try:
        excel_file = pd.ExcelFile(file_path)
        print(f"  Planilhas disponíveis: {excel_file.sheet_names}")

        processed_dfs_for_file = []
        for sheet_name in sheet_names_to_process:
            if sheet_name in excel_file.sheet_names:
                df_sheet = pd.read_excel(excel_file, sheet_name=sheet_name)
                df_sheet['Relatorio_Insumos'] = sheet_name
                df_sheet['ano_mes'] = current_ano_mes # Adicionar ano_mes específico do arquivo
                processed_dfs_for_file.append(df_sheet)
            else:
                print(f"  Warning: Planilha '{sheet_name}' não encontrada em '{file_path}'.")

        if processed_dfs_for_file:
            # Concatenar as planilhas do arquivo atual
            df_current_file = pd.concat(processed_dfs_for_file, ignore_index=True)
            # Limpar nomes das colunas
            df_current_file = clean_column_names(df_current_file)
            all_consolidated_dfs.append(df_current_file)
        else:
            print(f"  Nenhuma planilha processada para o arquivo '{file_path}'.")

    except FileNotFoundError:
        print(f"Erro: Arquivo não encontrado: {file_path}. Verifique se o arquivo está no diretório correto.")
    except Exception as e:
        print(f"Erro ao processar o arquivo {file_path}: {e}")

# 3. Concatenar todos os DataFrames processados em um único DataFrame final
if all_consolidated_dfs:
    df_consolidado = pd.concat(all_consolidated_dfs, ignore_index=True)
    print("\nProcessamento de todos os arquivos concluído com sucesso!")
    print(f"DataFrame consolidado criado com {len(df_consolidado)} linhas e {len(df_consolidado.columns)} colunas.")
    display(df_consolidado.head())
else:
    df_consolidado = pd.DataFrame()
    print("Nenhum dado foi processado ou consolidado.")

Processando arquivo: SINAPI_Referência_2025_01.xlsx
  Planilhas disponíveis: ['ISD', 'ICD', 'ISE', 'CSD', 'CCD', 'CSE', 'Analítico', 'Analítico com Custo']
Processando arquivo: SINAPI_Referência_2025_02.xlsx
  Planilhas disponíveis: ['ISD', 'ICD', 'ISE', 'CSD', 'CCD', 'CSE', 'Analítico', 'Analítico com Custo']
Processando arquivo: SINAPI_Referência_2025_03.xlsx
  Planilhas disponíveis: ['ISD', 'ICD', 'ISE', 'CSD', 'CCD', 'CSE', 'Analítico', 'Analítico com Custo']
Processando arquivo: SINAPI_Referência_2025_04.xlsx
  Planilhas disponíveis: ['ISD', 'ICD', 'ISE', 'CSD', 'CCD', 'CSE', 'Analítico', 'Analítico com Custo']
Processando arquivo: SINAPI_Referência_2025_05.xlsx
  Planilhas disponíveis: ['ISD', 'ICD', 'ISE', 'CSD', 'CCD', 'CSE', 'Analítico', 'Analítico com Custo']
Processando arquivo: SINAPI_Referência_2025_06.xlsx
  Planilhas disponíveis: ['ISD', 'ICD', 'ISE', 'CSD', 'CCD', 'CSE', 'Analítico', 'Analítico com Custo']
Processando arquivo: SINAPI_Referência_2025_07.xlsx
  Planilhas 

,classificação,código_doinsumo,descrição_do_insumo,unidade,origem_depreço,ac,al,am,ap,ba,...,rn,ro,rr,rs,sc,se,sp,to,relatorio_insumos,ano_mes
0,MATERIAL,11270,ABRACADEIRA DE LATAO PARA FIXACAO DE CABO PARA...,UN,CR,NaN,2.55,NaN,3.16,2.78,...,NaN,3.00,NaN,NaN,3.26,NaN,3.00,NaN,ISD,2025_01
1,MATERIAL,412,"ABRACADEIRA DE NYLON PARA AMARRACAO DE CABOS, ...",UN,CR,1.08,0.87,1.13,0.97,1.02,...,1.02,0.97,0.92,1.08,1.08,0.92,1.08,1.02,ISD,2025_01
2,MATERIAL,414,"ABRACADEIRA DE NYLON PARA AMARRACAO DE CABOS, ...",UN,CR,0.06,0.05,0.07,0.06,0.06,...,0.06,0.06,0.05,0.06,0.06,0.05,0.06,0.06,ISD,2025_01
3,MATERIAL,410,"ABRACADEIRA DE NYLON PARA AMARRACAO DE CABOS, ...",UN,CR,0.16,0.13,0.17,0.15,0.15,...,0.15,0.15,0.14,0.16,0.16,0.14,0.16,0.15,ISD,2025_01
4,MATERIAL,411,"ABRACADEIRA DE NYLON PARA AMARRACAO DE CABOS, ...",UN,C,0.21,0.17,0.22,0.19,0.20,...,0.20,0.19,0.18,0.21,0.21,0.18,0.21,0.20,ISD,2025_01


In [ ]:
df_consolidado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 277155 entries, 0 to 277154
Data columns (total 34 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   classificação        277155 non-null  object 
 1   código_doinsumo      277155 non-null  int64  
 2   descrição_do_insumo  277155 non-null  object 
 3   unidade              277155 non-null  object 
 4   origem_depreço       277155 non-null  object 
 5   ac                   190476 non-null  float64
 6   al                   184422 non-null  float64
 7   am                   155718 non-null  float64
 8   ap                   177288 non-null  float64
 9   ba                   243582 non-null  float64
 10  ce                   161250 non-null  float64
 11  df                   196791 non-null  float64
 12  es                   224727 non-null  float64
 13  go                   187560 non-null  float64
 14  ma                   174081 non-null  float64
 15  mg               

In [ ]:
df_consolidado.describe()

,código_doinsumo,ac,al,am,ap,ba,ce,df,es,go,...,pr,rj,rn,ro,rr,rs,sc,se,sp,to
count,277155.000000,1.904760e+05,184422.000000,1.557180e+05,1.772880e+05,2.435820e+05,1.612500e+05,196791.000000,2.247270e+05,187560.000000,...,1.938960e+05,2.017050e+05,192357.000000,2.235930e+05,145218.00000,2.427960e+05,2.072970e+05,1.652850e+05,2.771550e+05,1.837650e+05
mean,22449.726417,9.112647e+03,815.006969,5.498657e+03,1.922809e+03,2.025024e+04,3.527959e+03,954.565040,2.079895e+04,1385.765665,...,2.711596e+04,2.551389e+04,726.863414,2.246857e+04,1148.13486,2.703835e+04,2.806940e+04,3.608495e+03,3.557361e+04,4.443136e+03
std,16978.780953,2.252934e+05,7074.352997,9.067793e+04,3.587327e+04,2.757468e+05,5.189862e+04,8771.556832,3.160424e+05,16233.908746,...,3.498688e+05,3.437056e+05,7424.612219,3.295069e+05,9631.69495,3.300785e+05,3.537273e+05,4.675107e+04,3.718972e+05,6.048925e+04
min,1.000000,1.000000e-02,0.010000,1.000000e-02,1.000000e-02,1.000000e-02,1.000000e-02,0.010000,1.000000e-02,0.010000,...,1.000000e-02,1.000000e-02,0.010000,1.000000e-02,0.01000,1.000000e-02,1.000000e-02,1.000000e-02,1.000000e-02,1.000000e-02
25%,4730.000000,1.100000e+01,9.570000,8.760000e+00,9.190000e+00,1.108000e+01,8.470000e+00,10.700000,1.172000e+01,8.830000,...,9.460000e+00,1.010000e+01,9.270000,1.171000e+01,8.20000,1.333000e+01,1.006000e+01,8.450000e+00,1.252000e+01,9.810000e+00
50%,20146.000000,3.695000e+01,35.340000,3.016000e+01,2.965000e+01,4.201000e+01,2.943000e+01,36.610000,4.323000e+01,31.250000,...,3.218000e+01,3.919000e+01,32.000000,4.186000e+01,27.82000,4.563000e+01,3.615000e+01,3.034000e+01,4.624000e+01,3.570000e+01
75%,39394.000000,1.660900e+02,187.900000,1.509600e+02,1.479700e+02,2.200000e+02,1.540300e+02,191.840000,2.097100e+02,149.670000,...,1.712600e+02,2.107900e+02,154.220000,2.164900e+02,153.25000,2.451300e+02,1.771700e+02,1.513900e+02,2.413400e+02,1.822100e+02
max,45693.000000,1.007856e+07,290543.330000,4.618541e+06,1.831232e+06,9.275177e+06,1.654649e+06,300681.160000,9.864874e+06,804266.620000,...,9.859460e+06,1.040173e+07,290543.330000,1.073586e+07,290543.33000,1.106450e+07,1.040721e+07,1.607227e+06,1.099612e+07,2.742188e+06


In [ ]:
df_consolidado["relatorio_insumos"].unique() # valores únicos

array(['ISD', 'ICD', 'ISE'], dtype=object)

In [ ]:
df_consolidado["ano_mes"].unique() # valores únicos

array(['2025_01', '2025_02', '2025_03', '2025_04', '2025_05', '2025_06',
       '2025_07', '2025_08', '2025_09', '2025_10', '2025_11', '2025_12',
       '2026_01', '2026_02', '2026_03', '2026_04', '2026_05', '2026_06',
       '2026_07'], dtype=object)

In [ ]:
df_consolidado["relatorio_insumos"].value_counts() # frequência dos valores

,count
relatorio_insumos,
ISD,92385
ICD,92385
ISE,92385


In [ ]:
df_consolidado["ano_mes"].value_counts() # frequência dos valores

,count
ano_mes,
2025_09,14952
2025_06,14691
2025_08,14679
2025_07,14679
2026_07,14628
2026_06,14628
2026_05,14628
2025_05,14616
2026_02,14565


In [ ]:
#Carregar dados do Frame df_consolidado na tabela SINAPI_referencia_consolidada_jan25_jul26.xlsx disponível no google colab - ARQUIVO_B

output_file_name = ARQUIVO_B

try:
    df_consolidado.to_excel(output_file_name, index=False) # index=False para não salvar o índice do DataFrame como uma coluna
    print(f"DataFrame salvo com sucesso em '{output_file_name}'")
except Exception as e:
    print(f"Erro ao salvar o DataFrame: {e}")

Faz o download do arquivo chamado SINAPI_referencia_consolidada_jan25_jul26.xlsx que está salvo no diretório atual do Colab.

In [ ]:
from google.colab import files

files.download('SINAPI_referencia_consolidada_jan25_jul26.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



Esse código verifica se um arquivo existe e, se existir, o apaga. Caso contrário, informa que não foi encontrado.



In [ ]:
import os

if os.path.exists(ARQUIVO_B):
    os.remove(ARQUIVO_B)
    print(f"O arquivo {ARQUIVO_B} foi excluído com sucesso.")
else:    print(f"O arquivo {ARQUIVO_B} não foi encontrado.")

O arquivo SINAPI_referencia_consolidada_jan25_jul26.xlsx não foi encontrado.


In [ ]:
# Limpar os principais DataFrames e a lista de DataFrames
dfs = []
df_novos = pd.DataFrame() # Reinicia df_novos como um DataFrame vazio
# df_final = pd.DataFrame() # df_final é criado mais adiante, então não é necessário resetar aqui explicitamente se for sempre concatenado

print("Os DataFrames 'dfs' (lista) e 'df_novos' foram resetados.")
print("Você pode agora re-executar as células para carregar e processar novos dados.")

Os DataFrames 'dfs' (lista) e 'df_novos' foram resetados.
Você pode agora re-executar as células para carregar e processar novos dados.
